# Codebase Dependency Mapper — Claude-Code Harness over a Qdrant Code Store

A long-running code-analysis agent built on the same single-notebook Claude-Code
harness as `claude_code_from_scratch.ipynb`. Instead of acting on a live
filesystem, it reads a **codebase that has been chunked & embedded into a Qdrant
vector store**, walks every connection and dependency, and incrementally builds a
**dependency + external-connection map**.

It leans on the base harness's planning / reasoning / long-horizon machinery —
`todo_write` for the current pass, the persistent `task_*` graph for multi-session
coverage, rolling context compaction so it can grind through a large corpus, and
`spawn_subagent` to deep-analyze one file without polluting the lead's context.

### Tools added on top of the base harness

| Group | Tool | Purpose |
|---|---|---|
| **Read the store** | `qdrant_info` | Collection stats + auto-detect payload schema (text/path fields). Call this first. |
| | `list_files` | Inventory of distinct file paths (+ chunk counts) — the coverage checklist. |
| | `code_search` | Semantic search (embeds the query with `nomic-embed-text`). |
| | `code_keyword` | Deterministic substring sweep across chunks (e.g. `requests.`, `psycopg2`, `os.environ`). |
| | `code_scroll` | Paginate through *every* chunk for exhaustive coverage. |
| | `get_point` | Fetch one chunk's full code by id. |
| **Build the map** | `graph_add_edge` | Record an internal dependency (`imports`/`calls`/`inherits`/`references`). |
| | `graph_add_external` | Record an external connection (database, http_api, message_queue, cache, cloud_service, env_var, file_io, rpc, auth). |
| | `graph_add_node` | Annotate a node (kind/label/file). |
| | `graph_summary` | Current counts + highest fan-in/out. |
| | `graph_export` | Write `dependency_map.{json,dot,md}` to the sandbox. |

Plus the inherited `spawn_subagent`, `todo_*`, `task_*`, files/shell, skills, and
the `chat()` entry point.

### How it works
The agent embeds queries with Ollama (`nomic-embed-text`) and talks to Qdrant
over its REST API via raw `requests` (no `qdrant-client` dependency). The
dependency graph is a JSON file in the sandbox that survives restarts, so a
mapping run can be paused and resumed.

> **Set up before running:** edit the config cell — `QDRANT_URL`,
> `QDRANT_COLLECTION`, and (critically) `QDRANT_EMBED_MODEL`, which **must match
> the model used to index your corpus**, or semantic search returns garbage.
> `qdrant_info` validates the embedding dimension against the collection and warns
> on mismatch.


In [ ]:
"""
Imports + logging. One root logger "agent"; child loggers per subsystem.
Set AGENT_LOG_LEVEL=DEBUG for full payloads.
"""
from __future__ import annotations

import json
import logging
import os
import subprocess
import sys
import time
import uuid
import glob as _glob
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import requests  # talks to Ollama (chat + embeddings) and Qdrant REST

# Load any keys (QDRANT_URL/QDRANT_API_KEY/...) from the project .env if present.
try:
    from dotenv import load_dotenv, find_dotenv
    _dotenv = find_dotenv(usecwd=True)
    if _dotenv:
        load_dotenv(_dotenv)
except Exception:
    pass


AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()

class _Fmt(logging.Formatter):
    COLORS = {
        "DEBUG":    "\033[90m", "INFO": "\033[36m", "WARNING": "\033[33m",
        "ERROR":    "\033[31m", "CRITICAL": "\033[41m",
    }
    RESET = "\033[0m"
    def format(self, record: logging.LogRecord) -> str:
        color = self.COLORS.get(record.levelname, "")
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"{color}[{record.levelname:5s}] {short:18s} | {record.getMessage()}{self.RESET}"

_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())

log = logging.getLogger("agent")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False

log_loop    = logging.getLogger("agent.loop")
log_ollama  = logging.getLogger("agent.ollama")
log_tool    = logging.getLogger("agent.tool")
log_qdrant  = logging.getLogger("agent.qdrant")
log_graph   = logging.getLogger("agent.graph")
log_sub     = logging.getLogger("agent.subagent")
log_todo    = logging.getLogger("agent.todo")
log_compact = logging.getLogger("agent.compact")
log_chat    = logging.getLogger("agent.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")


In [ ]:
"""
Configuration. All knobs live here.
"""

# --- Ollama (chat + embeddings) ---------------------------------------------
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:8080")

MODELS = {
    "lead":       os.environ.get("AGENT_LEAD_MODEL",       "qwen3:32b"),
    "subagent":   os.environ.get("AGENT_SUBAGENT_MODEL",   "qwen3:8b"),
    "summarizer": os.environ.get("AGENT_SUMMARIZER_MODEL", "qwen3:8b"),
}

# --- Sandbox / working dir (where the map + state files are written) --------
WORKDIR = Path(os.environ.get("AGENT_WORKDIR", str(Path.cwd() / "sandbox"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
SKILLS_DIR = Path(os.environ.get("AGENT_SKILLS_DIR", str(WORKDIR / "skills")))

# --- State files (persisted across runs) ------------------------------------
TODO_FILE   = WORKDIR / ".agent_todo.json"
TASKS_FILE  = WORKDIR / ".agent_tasks.json"
MEMORY_FILE = WORKDIR / ".agent_memory.md"
GRAPH_FILE  = WORKDIR / "dependency_graph.json"   # the live, resumable map

# --- Limits ------------------------------------------------------------------
MAX_TOOL_OUTPUT     = 20_000
MAX_TURNS           = 40       # mapping a corpus is long-horizon
COMPRESS_THRESHOLD  = 45_000
KEEP_RECENT         = 6
BASH_TIMEOUT_S      = 60
REQUEST_TIMEOUT_S   = 600

# --- Qdrant ------------------------------------------------------------------
QDRANT_URL         = os.environ.get("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY     = os.environ.get("QDRANT_API_KEY")           # None for local
QDRANT_COLLECTION  = os.environ.get("QDRANT_COLLECTION", "code")
QDRANT_TIMEOUT_S   = 30
QDRANT_VECTOR_NAME = os.environ.get("QDRANT_VECTOR_NAME")       # None = unnamed/auto
QDRANT_SCAN_LIMIT  = int(os.environ.get("QDRANT_SCAN_LIMIT", "5000"))  # cap for keyword/list scans

# Embedding model for queries -- MUST match how the corpus was indexed.
QDRANT_EMBED_MODEL = os.environ.get("QDRANT_EMBED_MODEL", "nomic-embed-text")

# Payload field names. Leave blank to auto-detect from a sample (qdrant_info);
# set explicitly if auto-detection picks the wrong field.
QDRANT_TEXT_FIELD  = os.environ.get("QDRANT_TEXT_FIELD")  # e.g. "text" / "page_content"
QDRANT_PATH_FIELD  = os.environ.get("QDRANT_PATH_FIELD")  # e.g. "file_path" / "source"

BASH_BLOCKLIST = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "> /dev/", ":(){ :|:& };:"]

log.info(f"OLLAMA_HOST   = {OLLAMA_HOST}")
log.info(f"WORKDIR       = {WORKDIR}")
log.info(f"MODELS        = {MODELS}")
log.info(f"QDRANT_URL    = {QDRANT_URL}  collection={QDRANT_COLLECTION!r}")
log.info(f"EMBED_MODEL   = {QDRANT_EMBED_MODEL}")


In [ ]:
"""
Ollama client (chat). Thin wrapper over /api/chat with OpenAI-style tool calling.
"""

def ollama_chat(*, model: str, messages: List[Dict[str, Any]],
                tools: Optional[List[Dict[str, Any]]] = None,
                temperature: float = 0.2) -> Dict[str, Any]:
    payload: Dict[str, Any] = {
        "model": model, "messages": messages, "stream": False,
        "options": {"temperature": temperature},
    }
    if tools:
        payload["tools"] = tools

    approx = sum(len(str(m.get("content", ""))) for m in messages)
    log_ollama.info(f"-> {model}  msgs={len(messages)}  ~{approx} chars  "
                    f"tools={len(tools) if tools else 0}")

    t0 = time.time()
    resp = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=REQUEST_TIMEOUT_S)
    dt = time.time() - t0
    if resp.status_code != 200:
        log_ollama.error(f"<- HTTP {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

    data = resp.json()
    msg = data.get("message", {})
    tcs = msg.get("tool_calls") or []
    log_ollama.info(f"<- {model}  {dt:5.1f}s  text={len(msg.get('content') or '')}ch  "
                    f"tool_calls={len(tcs)}  done_reason={data.get('done_reason','?')}")
    return msg


def ollama_healthcheck() -> bool:
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
        tags = [m["name"] for m in r.json().get("models", [])]
        log_ollama.info(f"healthcheck OK -- {len(tags)} models available")
        for role, name in MODELS.items():
            present = any(t == name or t.startswith(name + ":") for t in tags)
            log_ollama.info(f"   {role:11s} {name:18s} [{'OK' if present else 'MISSING'}]")
        emb = any(t == QDRANT_EMBED_MODEL or t.startswith(QDRANT_EMBED_MODEL + ":") for t in tags)
        log_ollama.info(f"   embed       {QDRANT_EMBED_MODEL:18s} [{'OK' if emb else 'MISSING'}]")
        return True
    except Exception as e:
        log_ollama.error(f"healthcheck FAILED: {e}")
        return False

ollama_healthcheck()


In [ ]:
"""
Qdrant access layer (raw REST) + Ollama embeddings.

The code corpus lives in Qdrant as embedded chunks. We never need qdrant-client:
everything goes through the REST API with `requests`. Query embeddings come from
Ollama so they live in the same vector space as the indexed corpus (provided
QDRANT_EMBED_MODEL matches the indexing model).

Tools exposed to the agent:
  qdrant_info()                         -- stats + auto-detect payload schema
  list_files(path_prefix, max_files)    -- distinct file paths (+ chunk counts)
  code_search(query, limit, path_prefix)-- semantic (vector) search
  code_keyword(text, limit, path_prefix)-- deterministic substring sweep
  code_scroll(path_prefix, limit, offset)-- paginate every chunk
  get_point(point_id)                   -- one chunk's full code

`_truncate`, `MAX_TOOL_OUTPUT` come from the core-tools cell; they exist by call
time.
"""

# Detected-at-runtime collection facts.
VECTOR_SIZE: Optional[int] = None
VECTOR_NAME: Optional[str] = QDRANT_VECTOR_NAME
FIELDS: Dict[str, str] = {
    "text": QDRANT_TEXT_FIELD or "text",
    "path": QDRANT_PATH_FIELD or "file_path",
}
_FIELDS_LOCKED = {"text": bool(QDRANT_TEXT_FIELD), "path": bool(QDRANT_PATH_FIELD)}

_TEXT_CANDIDATES = ["text", "content", "page_content", "code", "document",
                    "chunk", "body", "source_code", "snippet"]
_PATH_CANDIDATES = ["file_path", "path", "filepath", "source", "file", "file_name",
                    "filename", "relative_path", "rel_path", "module"]


def embed_text(text: str) -> List[float]:
    """Embed one string via Ollama. Tries /api/embed then /api/embeddings."""
    try:
        r = requests.post(f"{OLLAMA_HOST}/api/embed",
                          json={"model": QDRANT_EMBED_MODEL, "input": text}, timeout=120)
        if r.status_code == 200:
            d = r.json()
            if d.get("embeddings"):
                return d["embeddings"][0]
            if d.get("embedding"):
                return d["embedding"]
    except Exception:
        pass
    r = requests.post(f"{OLLAMA_HOST}/api/embeddings",
                      json={"model": QDRANT_EMBED_MODEL, "prompt": text}, timeout=120)
    r.raise_for_status()
    return r.json()["embedding"]


def _qdrant(method: str, path: str, body: Optional[dict] = None) -> dict:
    headers = {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}
    r = requests.request(method, f"{QDRANT_URL}{path}", json=body,
                         headers=headers, timeout=QDRANT_TIMEOUT_S)
    if r.status_code != 200:
        raise RuntimeError(f"Qdrant HTTP {r.status_code}: {r.text[:300]}")
    return r.json()


def _detect_fields(sample_points: List[dict]) -> None:
    keys = set()
    for p in sample_points:
        keys.update((p.get("payload") or {}).keys())
    if not _FIELDS_LOCKED["text"]:
        for c in _TEXT_CANDIDATES:
            if c in keys:
                FIELDS["text"] = c
                break
    if not _FIELDS_LOCKED["path"]:
        for c in _PATH_CANDIDATES:
            if c in keys:
                FIELDS["path"] = c
                break


def _ensure_ready() -> Optional[str]:
    """Lazily detect vector size / name / payload fields. Returns error string or None."""
    global VECTOR_SIZE, VECTOR_NAME
    if VECTOR_SIZE is not None:
        return None
    try:
        info = _qdrant("GET", f"/collections/{QDRANT_COLLECTION}")["result"]
    except Exception as e:
        return f"Error: cannot reach Qdrant collection {QDRANT_COLLECTION!r} at {QDRANT_URL}: {e}"
    vectors = (info.get("config", {}).get("params", {}) or {}).get("vectors", {})
    if isinstance(vectors, dict) and "size" in vectors:
        VECTOR_SIZE = vectors["size"]
        VECTOR_NAME = None
    elif isinstance(vectors, dict) and vectors:
        name = QDRANT_VECTOR_NAME or next(iter(vectors))
        VECTOR_NAME = name
        VECTOR_SIZE = (vectors.get(name) or {}).get("size")
    try:
        sample = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/scroll",
                         {"limit": 5, "with_payload": True, "with_vector": False})["result"]["points"]
        _detect_fields(sample)
    except Exception:
        pass
    return None


def _coerce_id(v):
    s = str(v)
    try:
        return int(s)
    except ValueError:
        return s


def _payload(p: dict) -> dict:
    return p.get("payload") or {}


def _path_of(p: dict) -> str:
    return str(_payload(p).get(FIELDS["path"], ""))


def _text_of(p: dict) -> str:
    return str(_payload(p).get(FIELDS["text"], ""))


def _format_hits(points: List[dict], query: Optional[str] = None, snippet: int = 600) -> str:
    if not points:
        return "No results." + (f" (query: {query})" if query else "")
    out = []
    if query:
        out.append(f"{len(points)} result(s) for: {query}")
    for i, p in enumerate(points, 1):
        score = p.get("score")
        head = f"\n[{i}] id={p.get('id')}  path={_path_of(p) or '(no-path)'}"
        if isinstance(score, (int, float)):
            head += f"  score={score:.4f}"
        out.append(head)
        out.append(_truncate(_text_of(p).strip(), snippet))
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)


# ---- tools ------------------------------------------------------------------
def tool_qdrant_info() -> str:
    try:
        cols = _qdrant("GET", "/collections")["result"]["collections"]
    except Exception as e:
        return f"Error: cannot reach Qdrant at {QDRANT_URL}: {e}"
    names = [c["name"] for c in cols]
    out = [f"Qdrant: {QDRANT_URL}", f"Collections: {names}"]
    if QDRANT_COLLECTION not in names:
        out.append(f"\nWARNING: configured collection {QDRANT_COLLECTION!r} not found. "
                   f"Set QDRANT_COLLECTION in the config cell.")
        return "\n".join(out)
    err = _ensure_ready()
    if err:
        return err
    try:
        info = _qdrant("GET", f"/collections/{QDRANT_COLLECTION}")["result"]
        count = info.get("points_count")
    except Exception as e:
        return f"Error: {e}"
    out.append(f"\nCollection {QDRANT_COLLECTION!r}: points={count}  "
               f"vector_size={VECTOR_SIZE}  vector_name={VECTOR_NAME}")
    try:
        sample = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/scroll",
                         {"limit": 3, "with_payload": True, "with_vector": False})["result"]["points"]
    except Exception as e:
        sample = []
        out.append(f"(could not sample points: {e})")
    if sample:
        keys = sorted({k for p in sample for k in _payload(p)})
        out.append(f"Payload keys: {keys}")
        out.append(f"Detected text field: {FIELDS['text']!r}  |  path field: {FIELDS['path']!r}")
        out.append("Example payload (first point):")
        for k, v in _payload(sample[0]).items():
            out.append(f"  {k}: {str(v)[:120]}")
    # Embedding-dim sanity check.
    try:
        dim = len(embed_text("def f(): pass"))
        flag = "OK" if (VECTOR_SIZE is None or dim == VECTOR_SIZE) else "MISMATCH"
        out.append(f"\nEmbedding {QDRANT_EMBED_MODEL!r} dim={dim} vs collection {VECTOR_SIZE} [{flag}]")
        if flag == "MISMATCH":
            out.append("  -> semantic code_search will be unreliable. Set QDRANT_EMBED_MODEL to the "
                       "model used to index the corpus. (code_keyword/code_scroll still work.)")
    except Exception as e:
        out.append(f"\n(embedding probe failed: {e})")
    return _truncate("\n".join(out))


def tool_code_search(query: str, limit: int = 8, path_prefix: Optional[str] = None) -> str:
    log_qdrant.info(f"[search] {query[:80]!r} limit={limit} prefix={path_prefix!r}")
    err = _ensure_ready()
    if err:
        return err
    try:
        vec = embed_text(query)
    except Exception as e:
        return f"Error: embedding failed ({QDRANT_EMBED_MODEL}): {e}"
    if VECTOR_SIZE and len(vec) != VECTOR_SIZE:
        return (f"Error: embedding dim {len(vec)} != collection vector size {VECTOR_SIZE}. "
                f"QDRANT_EMBED_MODEL={QDRANT_EMBED_MODEL!r} does not match the corpus index. "
                f"Use code_keyword/code_scroll instead, or fix the model.")
    fetch = limit * 5 if path_prefix else limit
    body: Dict[str, Any] = {"limit": fetch, "with_payload": True, "with_vector": False}
    body["vector"] = {"name": VECTOR_NAME, "vector": vec} if VECTOR_NAME else vec
    try:
        res = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/search", body)["result"]
    except Exception as e:
        return f"Error: search failed: {e}"
    if path_prefix:
        res = [p for p in res if _path_of(p).startswith(path_prefix)][:limit]
    else:
        res = res[:limit]
    return _format_hits(res, query=query)


def tool_code_keyword(text: str, limit: int = 20, path_prefix: Optional[str] = None) -> str:
    log_qdrant.info(f"[keyword] {text[:60]!r} limit={limit} prefix={path_prefix!r}")
    err = _ensure_ready()
    if err:
        return err
    needle = text.lower()
    matches: List[dict] = []
    scanned = 0
    offset = None
    while scanned < QDRANT_SCAN_LIMIT and len(matches) < limit:
        batch = min(256, QDRANT_SCAN_LIMIT - scanned)
        body: Dict[str, Any] = {"limit": batch, "with_payload": True, "with_vector": False}
        if offset is not None:
            body["offset"] = offset
        try:
            r = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/scroll", body)["result"]
        except Exception as e:
            return f"Error: scroll failed: {e}"
        pts = r.get("points", [])
        offset = r.get("next_page_offset")
        for p in pts:
            scanned += 1
            if path_prefix and not _path_of(p).startswith(path_prefix):
                continue
            if needle in _text_of(p).lower():
                matches.append(p)
                if len(matches) >= limit:
                    break
        if not offset:
            break
    if not matches:
        return f"No chunks containing {text!r} found (scanned {scanned} points)."
    out = [f"{len(matches)} chunk(s) containing {text!r} (scanned {scanned}):"]
    for i, p in enumerate(matches, 1):
        out.append(f"\n[{i}] id={p.get('id')}  path={_path_of(p) or '(no-path)'}")
        # show the matching lines for context
        hit_lines = [ln for ln in _text_of(p).splitlines() if needle in ln.lower()][:6]
        out.append("\n".join(f"    {ln.strip()}" for ln in hit_lines) or "    (match in body)")
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)


def tool_code_scroll(path_prefix: Optional[str] = None, limit: int = 40,
                     offset: Optional[Any] = None) -> str:
    log_qdrant.info(f"[scroll] prefix={path_prefix!r} limit={limit} offset={offset!r}")
    err = _ensure_ready()
    if err:
        return err
    body: Dict[str, Any] = {"limit": limit, "with_payload": True, "with_vector": False}
    if offset not in (None, "", "None"):
        body["offset"] = offset
    try:
        r = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/scroll", body)["result"]
    except Exception as e:
        return f"Error: scroll failed: {e}"
    pts = r.get("points", [])
    nxt = r.get("next_page_offset")
    if path_prefix:
        pts = [p for p in pts if _path_of(p).startswith(path_prefix)]
    out = [f"{len(pts)} chunk(s).  next_offset={nxt!r}  "
           f"(pass this back as offset= to continue; null means end)."]
    for p in pts:
        out.append(f"\nid={p.get('id')}  path={_path_of(p) or '(no-path)'}")
        out.append(_truncate(_text_of(p).strip(), 300))
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)


def tool_list_files(path_prefix: Optional[str] = None, max_files: int = 500) -> str:
    log_qdrant.info(f"[list_files] prefix={path_prefix!r}")
    err = _ensure_ready()
    if err:
        return err
    counts: Dict[str, int] = {}
    scanned = 0
    offset = None
    while scanned < QDRANT_SCAN_LIMIT:
        body: Dict[str, Any] = {"limit": 256, "with_payload": True, "with_vector": False}
        if offset is not None:
            body["offset"] = offset
        try:
            r = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points/scroll", body)["result"]
        except Exception as e:
            return f"Error: scroll failed: {e}"
        pts = r.get("points", [])
        offset = r.get("next_page_offset")
        for p in pts:
            scanned += 1
            path = _path_of(p)
            if not path:
                continue
            if path_prefix and not path.startswith(path_prefix):
                continue
            counts[path] = counts.get(path, 0) + 1
        if not offset:
            break
    if not counts:
        return f"No files found (scanned {scanned}; path field={FIELDS['path']!r})."
    files = sorted(counts)
    shown = files[:max_files]
    out = [f"{len(files)} distinct file(s) across {scanned} chunks "
           f"(showing {len(shown)}):"]
    out += [f"  {counts[f]:4d}  {f}" for f in shown]
    if len(files) > max_files:
        out.append(f"  ... {len(files) - max_files} more (raise max_files or use path_prefix).")
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)


def tool_get_point(point_id: Any) -> str:
    log_qdrant.info(f"[get_point] {point_id}")
    err = _ensure_ready()
    if err:
        return err
    body = {"ids": [_coerce_id(point_id)], "with_payload": True, "with_vector": False}
    try:
        res = _qdrant("POST", f"/collections/{QDRANT_COLLECTION}/points", body)["result"]
    except Exception as e:
        return f"Error: retrieve failed: {e}"
    if not res:
        return f"No point with id {point_id}."
    p = res[0]
    pl = _payload(p)
    out = [f"Point id={p.get('id')}", f"path: {pl.get(FIELDS['path'])}"]
    for k, v in pl.items():
        if k == FIELDS["text"]:
            continue
        out.append(f"{k}: {str(v)[:200]}")
    out.append("---- code ----")
    out.append(_text_of(p))
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)

log.info("Qdrant tools defined: qdrant_info, list_files, code_search, code_keyword, code_scroll, get_point")


In [ ]:
"""
Dependency-graph store + tools.

The map is a JSON document in the sandbox (GRAPH_FILE) -- so it is durable and a
mapping run can be paused/resumed. Shape:

  {
    "nodes":     {node_id: {"kind": ..., "label": ..., "file": ...}},
    "edges":     [{"source","target","type","evidence":[...]}],
    "externals": [{"component","kind","target","details","evidence":[...]}]
  }

node kinds : module | file | package | external | symbol
edge types : imports | calls | inherits | references | uses
external kinds: database | http_api | message_queue | cache | cloud_service
              | env_var | file_io | rpc | auth | other

Tools (de-duplicated; repeated adds merge evidence rather than pile up):
  graph_add_node(id, kind, label, file)
  graph_add_edge(source, target, type, evidence)
  graph_add_external(component, kind, target, details, evidence)
  graph_summary()
  graph_export(format)   format = json | dot | markdown | all
"""

EDGE_TYPES = ("imports", "calls", "inherits", "references", "uses")
EXTERNAL_KINDS = ("database", "http_api", "message_queue", "cache", "cloud_service",
                  "env_var", "file_io", "rpc", "auth", "other")


def _empty_graph() -> Dict[str, Any]:
    return {"nodes": {}, "edges": [], "externals": []}


def _load_graph() -> Dict[str, Any]:
    if not GRAPH_FILE.exists():
        return _empty_graph()
    try:
        g = json.loads(GRAPH_FILE.read_text(encoding="utf-8"))
        g.setdefault("nodes", {}); g.setdefault("edges", []); g.setdefault("externals", [])
        return g
    except (json.JSONDecodeError, OSError) as e:
        log_graph.warning(f"graph file unreadable, starting empty: {e}")
        return _empty_graph()


def _save_graph(g: Dict[str, Any]) -> None:
    GRAPH_FILE.write_text(json.dumps(g, indent=2), encoding="utf-8")


def _ensure_node(g: Dict[str, Any], nid: str, kind: str = "module") -> None:
    if nid not in g["nodes"]:
        g["nodes"][nid] = {"kind": kind, "label": nid, "file": None}


def run_graph_add_node(node_id: str, kind: str = "module",
                       label: Optional[str] = None, file: Optional[str] = None) -> str:
    g = _load_graph()
    n = g["nodes"].get(node_id, {"kind": kind, "label": node_id, "file": None})
    n["kind"] = kind or n.get("kind", "module")
    if label:
        n["label"] = label
    if file:
        n["file"] = file
    g["nodes"][node_id] = n
    _save_graph(g)
    log_graph.info(f"[node] {node_id} ({kind})")
    return f"node: {node_id} [{n['kind']}]"


def run_graph_add_edge(source: str, target: str, type: str = "imports",
                       evidence: str = "") -> str:
    if type not in EDGE_TYPES:
        return f"Error: type must be one of {EDGE_TYPES}, got {type!r}"
    g = _load_graph()
    _ensure_node(g, source)
    _ensure_node(g, target)
    for e in g["edges"]:
        if e["source"] == source and e["target"] == target and e["type"] == type:
            if evidence and evidence not in e["evidence"]:
                e["evidence"].append(evidence)
            _save_graph(g)
            return f"edge exists (evidence merged): {source} -[{type}]-> {target}"
    g["edges"].append({"source": source, "target": target, "type": type,
                       "evidence": [evidence] if evidence else []})
    _save_graph(g)
    log_graph.info(f"[edge] {source} -[{type}]-> {target}")
    return f"added edge: {source} -[{type}]-> {target}"


def run_graph_add_external(component: str, kind: str, target: str,
                           details: str = "", evidence: str = "") -> str:
    if kind not in EXTERNAL_KINDS:
        return f"Error: kind must be one of {EXTERNAL_KINDS}, got {kind!r}"
    g = _load_graph()
    _ensure_node(g, component)
    for x in g["externals"]:
        if x["component"] == component and x["kind"] == kind and x["target"] == target:
            if details and not x.get("details"):
                x["details"] = details
            if evidence and evidence not in x["evidence"]:
                x["evidence"].append(evidence)
            _save_graph(g)
            return f"external exists (merged): {component} --{kind}--> {target}"
    g["externals"].append({"component": component, "kind": kind, "target": target,
                           "details": details, "evidence": [evidence] if evidence else []})
    _save_graph(g)
    log_graph.info(f"[external] {component} --{kind}--> {target}")
    return f"added external: {component} --{kind}--> {target}"


def run_graph_summary() -> str:
    g = _load_graph()
    nodes, edges, externals = g["nodes"], g["edges"], g["externals"]
    by_kind: Dict[str, int] = {}
    for n in nodes.values():
        by_kind[n.get("kind", "?")] = by_kind.get(n.get("kind", "?"), 0) + 1
    by_etype: Dict[str, int] = {}
    for e in edges:
        by_etype[e["type"]] = by_etype.get(e["type"], 0) + 1
    by_xkind: Dict[str, int] = {}
    for x in externals:
        by_xkind[x["kind"]] = by_xkind.get(x["kind"], 0) + 1
    fan_out: Dict[str, int] = defaultdict(int)
    fan_in: Dict[str, int] = defaultdict(int)
    for e in edges:
        fan_out[e["source"]] += 1
        fan_in[e["target"]] += 1
    top_out = sorted(fan_out.items(), key=lambda kv: -kv[1])[:5]
    top_in = sorted(fan_in.items(), key=lambda kv: -kv[1])[:5]
    out = [
        f"nodes={len(nodes)} {dict(by_kind)}",
        f"edges={len(edges)} {dict(by_etype)}",
        f"externals={len(externals)} {dict(by_xkind)}",
        f"top fan-out (depends on most): {top_out}",
        f"top fan-in (most depended-on): {top_in}",
    ]
    return "\n".join(out)


def _esc_dot(s: Any) -> str:
    return str(s).replace("\\", "/").replace('"', "'").replace("\n", " ")


def _to_dot(g: Dict[str, Any]) -> str:
    lines = ["digraph dependencies {", "  rankdir=LR;",
             "  node [fontsize=10];", "  edge [fontsize=8];"]
    for nid, n in g["nodes"].items():
        kind = n.get("kind", "module")
        shape = "box" if kind in ("module", "file", "package") else "ellipse"
        lines.append(f'  "{_esc_dot(nid)}" [shape={shape}, label="{_esc_dot(n.get("label") or nid)}"];')
    for e in g["edges"]:
        lines.append(f'  "{_esc_dot(e["source"])}" -> "{_esc_dot(e["target"])}" '
                     f'[label="{_esc_dot(e["type"])}"];')
    for i, x in enumerate(g["externals"]):
        ext_id = f'ext::{x["kind"]}::{x["target"]}::{i}'
        lines.append(f'  "{_esc_dot(ext_id)}" [shape=box, style=filled, fillcolor=lightgrey, '
                     f'label="{_esc_dot(x["kind"])}: {_esc_dot(x["target"])}"];')
        lines.append(f'  "{_esc_dot(x["component"])}" -> "{_esc_dot(ext_id)}" '
                     f'[style=dashed, label="{_esc_dot(x["kind"])}"];')
    lines.append("}")
    return "\n".join(lines)


def _md_cell(s: Any) -> str:
    return str(s).replace("|", "/").replace("\n", " ")[:200]


def _to_markdown(g: Dict[str, Any]) -> str:
    nodes, edges, externals = g["nodes"], g["edges"], g["externals"]
    by_kind: Dict[str, int] = {}
    for n in nodes.values():
        by_kind[n.get("kind", "?")] = by_kind.get(n.get("kind", "?"), 0) + 1
    by_xkind: Dict[str, int] = {}
    for x in externals:
        by_xkind[x["kind"]] = by_xkind.get(x["kind"], 0) + 1
    L = ["# Dependency & External-Connection Map", "",
         f"- **Nodes:** {len(nodes)} {dict(by_kind)}",
         f"- **Internal edges:** {len(edges)}",
         f"- **External connections:** {len(externals)} {dict(by_xkind)}", "",
         "## External connections", ""]
    if externals:
        L += ["| Component | Kind | Target | Details | Evidence |",
              "|---|---|---|---|---|"]
        for x in sorted(externals, key=lambda z: (z["kind"], z["component"])):
            ev = "; ".join(x.get("evidence", []))
            L.append(f"| {_md_cell(x['component'])} | {x['kind']} | {_md_cell(x['target'])} "
                     f"| {_md_cell(x.get('details', ''))} | {_md_cell(ev)} |")
    else:
        L.append("_None recorded yet._")
    L += ["", "## Internal dependencies", ""]
    by_src = defaultdict(list)
    for e in edges:
        by_src[e["source"]].append(e)
    if by_src:
        for src in sorted(by_src):
            L.append(f"### {src}")
            for e in by_src[src]:
                L.append(f"- {e['type']} -> `{e['target']}`")
            L.append("")
    else:
        L.append("_None recorded yet._")
    return "\n".join(L)


def run_graph_export(format: str = "all") -> str:
    g = _load_graph()
    fmt = (format or "all").lower()
    written = []
    if fmt in ("json", "all"):
        p = WORKDIR / "dependency_map.json"
        p.write_text(json.dumps(g, indent=2), encoding="utf-8")
        written.append(str(p))
    if fmt in ("dot", "all"):
        p = WORKDIR / "dependency_map.dot"
        p.write_text(_to_dot(g), encoding="utf-8")
        written.append(str(p))
    if fmt in ("markdown", "md", "report", "all"):
        p = WORKDIR / "dependency_map.md"
        p.write_text(_to_markdown(g), encoding="utf-8")
        written.append(str(p))
    if not written:
        return f"Error: unknown format {format!r}. Use json | dot | markdown | all."
    log_graph.info(f"[export] {fmt} -> {written}")
    return ("Exported:\n" + "\n".join(f"  {w}" for w in written)
            + "\n\n--- summary ---\n" + run_graph_summary())

log.info("Graph tools defined: graph_add_node/edge/external, graph_summary, graph_export")


In [ ]:
"""
Core file & shell tools (sandboxed to WORKDIR). Let the agent persist reports,
intermediate notes, and run graphviz/jq on the exported map.
"""

SNAPSHOTS: Dict[str, Optional[str]] = {}


def _safe_path(path: str) -> Path:
    p = (WORKDIR / path).resolve() if not os.path.isabs(path) else Path(path).resolve()
    try:
        p.relative_to(WORKDIR)
    except ValueError:
        raise ValueError(f"path escapes WORKDIR: {p}")
    return p


def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    if len(s) <= limit:
        return s
    return s[:limit] + f"\n... [truncated {len(s) - limit} chars]"


def tool_bash(command: str) -> str:
    log_tool.info(f"[bash] {command[:120]}")
    for bad in BASH_BLOCKLIST:
        if bad in command:
            log_tool.warning(f"[bash] BLOCKED ({bad!r})")
            return f"Error: blocked by safety policy (matched {bad!r})"
    try:
        proc = subprocess.run(command, shell=True, cwd=str(WORKDIR),
                              capture_output=True, text=True, timeout=BASH_TIMEOUT_S)
        out = (proc.stdout + proc.stderr).strip() or "(no output)"
        log_tool.info(f"[bash] exit={proc.returncode} bytes={len(out)}")
        return _truncate(out)
    except subprocess.TimeoutExpired:
        return f"Error: timeout after {BASH_TIMEOUT_S}s"
    except Exception as e:
        return f"Error: {e}"


def tool_read(path: str, start_line: Optional[int] = None, end_line: Optional[int] = None) -> str:
    log_tool.info(f"[read] {path} lines={start_line}:{end_line}")
    try:
        p = _safe_path(path)
        lines = p.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
        i0 = max(0, (start_line or 1) - 1)
        i1 = end_line if end_line is not None else len(lines)
        numbered = "".join(f"{i0+1+i:5d}\t{ln}" for i, ln in enumerate(lines[i0:i1]))
        return _truncate(numbered or "(empty)")
    except FileNotFoundError:
        return f"Error: file not found: {path}"
    except Exception as e:
        return f"Error: {e}"


def tool_write(path: str, content: str) -> str:
    log_tool.info(f"[write] {path} ({len(content)} chars)")
    try:
        p = _safe_path(path)
        if p.exists():
            SNAPSHOTS[str(p)] = p.read_text(encoding="utf-8", errors="replace")
            action = "updated"
        else:
            SNAPSHOTS[str(p)] = None
            action = "created"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        return f"{action}: {p} (snapshot saved -- use revert to undo)"
    except Exception as e:
        return f"Error: {e}"


def tool_grep(pattern: str, path: str = ".", recursive: bool = True) -> str:
    log_tool.info(f"[grep] {pattern!r} in {path}")
    try:
        p = _safe_path(path)
        flags = ["-rn"] if recursive else ["-n"]
        proc = subprocess.run(["grep", *flags, pattern, str(p)],
                              capture_output=True, text=True, timeout=30)
        out = (proc.stdout + proc.stderr).strip() or "(no matches)"
        return _truncate(out, 10_000)
    except subprocess.TimeoutExpired:
        return "Error: grep timeout"
    except Exception as e:
        return f"Error: {e}"


def tool_glob(pattern: str) -> str:
    log_tool.info(f"[glob] {pattern}")
    full = str(WORKDIR / pattern) if not os.path.isabs(pattern) else pattern
    matches = sorted(_glob.glob(full, recursive=True))[:200]
    safe = [m for m in matches if Path(m).resolve().is_relative_to(WORKDIR)]
    return "\n".join(safe) if safe else "(no matches)"


def tool_revert(path: str) -> str:
    log_tool.info(f"[revert] {path}")
    try:
        p = _safe_path(path)
        key = str(p)
        if key not in SNAPSHOTS:
            return f"Error: no snapshot for {p}"
        prev = SNAPSHOTS.pop(key)
        if prev is None:
            p.unlink(missing_ok=True)
            return f"reverted: deleted {p} (was a new file)"
        p.write_text(prev, encoding="utf-8")
        return f"reverted: {p}"
    except Exception as e:
        return f"Error: {e}"

log.info("Core tools defined: bash, read, write, grep, glob, revert")


In [ ]:
"""
Tool schemas + base dispatch.

BASE = read-the-store + build-the-map + file/shell. Subagents get exactly this
(so a subagent can analyse a file AND write its findings into the shared graph).
The LEAD registry (later cells) adds spawn_subagent, planning and skills.
"""

def _fn(name, description, properties, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required}}}


TOOLS_BASE: List[Dict[str, Any]] = [
    # ---- read the Qdrant code store ----
    _fn("qdrant_info",
        "Inspect the Qdrant code collection: point count, vector size, detected payload "
        "fields (which key holds the code text vs the file path), an example payload, and "
        "an embedding-dimension sanity check. CALL THIS FIRST.",
        {}, []),
    _fn("list_files",
        "List distinct file paths in the corpus with their chunk counts -- your coverage "
        "checklist. Optionally restrict to a path prefix.",
        {"path_prefix": {"type": "string", "description": "Only paths starting with this."},
         "max_files":   {"type": "integer", "description": "Default 500."}},
        []),
    _fn("code_search",
        "Semantic (vector) search over the code corpus -- finds chunks related in meaning "
        "to the query (e.g. 'database connection setup', 'http client', 'message queue "
        "consumer'). Returns chunk ids, paths and snippets.",
        {"query":       {"type": "string"},
         "limit":       {"type": "integer", "description": "Default 8."},
         "path_prefix": {"type": "string", "description": "Restrict to paths with this prefix."}},
        ["query"]),
    _fn("code_keyword",
        "Deterministic substring sweep across ALL chunks (e.g. 'import ', 'requests.', "
        "'psycopg2', 'boto3', 'os.environ', 'KafkaConsumer'). Use to exhaustively find "
        "every occurrence of a dependency/connection marker. Shows the matching lines.",
        {"text":        {"type": "string", "description": "Exact substring to find (case-insensitive)."},
         "limit":       {"type": "integer", "description": "Default 20."},
         "path_prefix": {"type": "string"}},
        ["text"]),
    _fn("code_scroll",
        "Paginate through chunks in storage order for exhaustive coverage. Pass the "
        "returned next_offset back as 'offset' to continue; a null next_offset means the "
        "end. Optionally restrict to a path prefix.",
        {"path_prefix": {"type": "string"},
         "limit":       {"type": "integer", "description": "Default 40."},
         "offset":      {"type": "string",  "description": "next_offset from a previous call."}},
        []),
    _fn("get_point",
        "Fetch one chunk's FULL code and payload by its point id (ids come from the other "
        "tools).",
        {"point_id": {"type": "string"}},
        ["point_id"]),
    # ---- build the dependency map ----
    _fn("graph_add_edge",
        "Record an INTERNAL dependency between two modules/files/symbols in the corpus. "
        "type is one of imports|calls|inherits|references|uses. Include evidence (the "
        "import/call line).",
        {"source":   {"type": "string", "description": "The dependent module/file/symbol."},
         "target":   {"type": "string", "description": "The depended-on module/file/symbol."},
         "type":     {"type": "string", "enum": list(EDGE_TYPES)},
         "evidence": {"type": "string", "description": "The line/snippet that proves it."}},
        ["source", "target"]),
    _fn("graph_add_external",
        "Record an EXTERNAL connection: a component talking to something outside the "
        "codebase. kind is one of database|http_api|message_queue|cache|cloud_service|"
        "env_var|file_io|rpc|auth|other.",
        {"component": {"type": "string", "description": "The module/file making the connection."},
         "kind":      {"type": "string", "enum": list(EXTERNAL_KINDS)},
         "target":    {"type": "string", "description": "What it connects to (e.g. 'PostgreSQL', "
                       "'https://api.stripe.com', 'KAFKA_BROKER env', 'AWS S3')."},
         "details":   {"type": "string", "description": "Driver/library, host, etc."},
         "evidence":  {"type": "string", "description": "The line/snippet that proves it."}},
        ["component", "kind", "target"]),
    _fn("graph_add_node",
        "Annotate a node's kind/label/file (module|file|package|external|symbol). Optional; "
        "edges/externals auto-create nodes.",
        {"node_id": {"type": "string"},
         "kind":    {"type": "string", "enum": ["module", "file", "package", "external", "symbol"]},
         "label":   {"type": "string"},
         "file":    {"type": "string"}},
        ["node_id"]),
    _fn("graph_summary",
        "Show current map stats: node/edge/external counts by kind, plus highest fan-in/out nodes.",
        {}, []),
    _fn("graph_export",
        "Write the map to the sandbox. format = json | dot | markdown | all. Returns the "
        "summary too. Call when a pass is complete.",
        {"format": {"type": "string", "enum": ["json", "dot", "markdown", "all"]}},
        []),
    # ---- file/shell ----
    _fn("bash", "Run a shell command in the sandbox WORKDIR (e.g. graphviz `dot`, `jq`).",
        {"command": {"type": "string"}}, ["command"]),
    _fn("read", "Read a sandbox file (optional 1-indexed line range).",
        {"path": {"type": "string"}, "start_line": {"type": "integer"}, "end_line": {"type": "integer"}},
        ["path"]),
    _fn("write", "Write a sandbox file (e.g. a notes/report file). Snapshotted; revert to undo.",
        {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"]),
    _fn("grep", "Regex search across sandbox files.",
        {"pattern": {"type": "string"}, "path": {"type": "string"}, "recursive": {"type": "boolean"}},
        ["pattern"]),
    _fn("glob", "Find sandbox files matching a glob, e.g. '**/*.md'.",
        {"pattern": {"type": "string"}}, ["pattern"]),
    _fn("revert", "Restore a sandbox file to its state before the most recent write.",
        {"path": {"type": "string"}}, ["path"]),
]

DISPATCH_BASE: Dict[str, Callable[[Dict[str, Any]], str]] = {
    "qdrant_info":  lambda a: tool_qdrant_info(),
    "list_files":   lambda a: tool_list_files(a.get("path_prefix"), a.get("max_files", 500)),
    "code_search":  lambda a: tool_code_search(a["query"], a.get("limit", 8), a.get("path_prefix")),
    "code_keyword": lambda a: tool_code_keyword(a["text"], a.get("limit", 20), a.get("path_prefix")),
    "code_scroll":  lambda a: tool_code_scroll(a.get("path_prefix"), a.get("limit", 40), a.get("offset")),
    "get_point":    lambda a: tool_get_point(a["point_id"]),
    "graph_add_edge":     lambda a: run_graph_add_edge(a["source"], a["target"], a.get("type", "imports"), a.get("evidence", "")),
    "graph_add_external": lambda a: run_graph_add_external(a["component"], a["kind"], a["target"], a.get("details", ""), a.get("evidence", "")),
    "graph_add_node":     lambda a: run_graph_add_node(a["node_id"], a.get("kind", "module"), a.get("label"), a.get("file")),
    "graph_summary":      lambda a: run_graph_summary(),
    "graph_export":       lambda a: run_graph_export(a.get("format", "all")),
    "bash":   lambda a: tool_bash(a["command"]),
    "read":   lambda a: tool_read(a["path"], a.get("start_line"), a.get("end_line")),
    "write":  lambda a: tool_write(a["path"], a["content"]),
    "grep":   lambda a: tool_grep(a["pattern"], a.get("path", "."), a.get("recursive", True)),
    "glob":   lambda a: tool_glob(a["pattern"]),
    "revert": lambda a: tool_revert(a["path"]),
}

log.info(f"Base tool registry: {list(DISPATCH_BASE)}")


In [ ]:
"""
Agent loop -- perception -> action -> observation -> repeat. Terminates when the
model returns a message with no tool_calls. MAX_TURNS caps runaway loops;
`messages` is mutated in place.
"""

def _run_one_tool_call(tc: Dict[str, Any], dispatch: Dict[str, Callable]) -> Dict[str, Any]:
    fn   = tc.get("function", {}) or {}
    name = fn.get("name", "")
    args = fn.get("arguments", {}) or {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}
    log_tool.info(f"-> {name}(" + ", ".join(f"{k}={str(v)[:40]!r}" for k, v in args.items()) + ")")
    if name not in dispatch:
        result = f"[error] Unknown tool: {name!r}. Available: {sorted(dispatch)}"
        log_tool.warning(result)
    else:
        try:
            result = dispatch[name](args)
        except Exception as e:
            result = f"[error] Tool {name!r} raised {type(e).__name__}: {e}"
            log_tool.exception(result)
    result = _truncate(str(result), MAX_TOOL_OUTPUT)
    return {"role": "tool", "name": name, "content": result}


def agent_loop(messages, tools, dispatch, model=MODELS["lead"],
               max_turns=MAX_TURNS, label="lead") -> str:
    tool_names = [t["function"]["name"] for t in tools]
    log_loop.info(f"[{label}] starting (model={model}, max_turns={max_turns}, tools={len(tool_names)})")
    final_text = ""
    for turn in range(1, max_turns + 1):
        log_loop.info(f"[{label}] turn {turn}/{max_turns}  msgs={len(messages)}")
        msg = ollama_chat(model=model, messages=messages, tools=tools)
        messages.append(msg)
        tool_calls = msg.get("tool_calls") or []
        text = (msg.get("content") or "").strip()
        if not tool_calls:
            final_text = text
            log_loop.info(f"[{label}] DONE on turn {turn} -- no tool_calls.")
            return final_text
        log_loop.info(f"[{label}] turn {turn}: {len(tool_calls)} tool call(s)")
        for tc in tool_calls:
            messages.append(_run_one_tool_call(tc, dispatch))
    log_loop.warning(f"[{label}] hit MAX_TURNS={max_turns} without termination.")
    return final_text or "[agent_loop] max turns reached without a terminal response"

log.info("Agent loop ready.")


In [ ]:
"""
Subagents -- isolated context for deep single-file analysis.

When the lead wants to fully analyse one file/module (read every chunk, trace
its imports & connections) it calls spawn_subagent. The subagent runs a fresh
loop on the smaller model with the BASE tools -- which include the Qdrant read
tools AND the graph-write tools, so it can record edges/externals directly into
the shared map. Only its final summary returns to the lead.
"""

SUBAGENT_SYSTEM = (
    "You are a focused code-analysis subagent. The codebase lives in a Qdrant vector store; "
    "use qdrant_info/list_files/code_search/code_keyword/code_scroll/get_point to read it. "
    "You are given ONE file or module to analyse exhaustively. Read all of its chunks, then "
    "identify (a) INTERNAL dependencies -- which other modules/files it imports, calls, "
    "inherits from or references -- and (b) EXTERNAL connections -- databases, HTTP/REST "
    "APIs, message queues, caches, cloud services, environment variables, file I/O, RPC, "
    "auth providers. Record each finding immediately with graph_add_edge / graph_add_external, "
    "always citing the exact line as evidence. Never invent a dependency; only record what is "
    "in the code. When finished, reply with a concise list of what you recorded. Do not ask "
    "questions; make reasonable assumptions and proceed."
)


def run_subagent(prompt: str) -> str:
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- {prompt[:120]!r}")
    sub_messages = [{"role": "system", "content": SUBAGENT_SYSTEM},
                    {"role": "user", "content": prompt}]
    try:
        result = agent_loop(sub_messages, TOOLS_BASE, DISPATCH_BASE,
                            model=MODELS["subagent"], label=f"sub:{sub_id}")
    except Exception as e:
        log_sub.exception(f"[sub:{sub_id}] crashed: {e}")
        return f"[subagent error] {type(e).__name__}: {e}"
    log_sub.info(f"[sub:{sub_id}] done -- {len(result)} chars")
    return result


TOOLS_LEAD: List[Dict[str, Any]] = TOOLS_BASE + [
    _fn("spawn_subagent",
        "Delegate exhaustive analysis of ONE file/module to a fresh subagent (it has the "
        "same Qdrant + graph tools and writes directly into the shared map). Use this to "
        "keep your own context clean while covering the corpus file by file. Returns the "
        "subagent's summary of what it recorded.",
        {"prompt": {"type": "string",
                    "description": "Name the exact file/module and instruct the subagent to map "
                    "its internal deps and external connections into the graph."}},
        ["prompt"])]

DISPATCH_LEAD: Dict[str, Callable[[Dict[str, Any]], str]] = {
    **DISPATCH_BASE, "spawn_subagent": lambda a: run_subagent(a["prompt"])}

log.info(f"Subagent ready. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Todo planning -- the working checklist for the CURRENT pass. Persisted to TODO_FILE.
"""

_TODO_STATUSES = ("pending", "in_progress", "done")

def _load_todos():
    if not TODO_FILE.exists():
        return []
    try:
        return json.loads(TODO_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []

def _save_todos(items):
    TODO_FILE.write_text(json.dumps(items, indent=2), encoding="utf-8")

def run_todo_write(items):
    todos = [{"index": i, "text": t, "status": "pending"} for i, t in enumerate(items)]
    _save_todos(todos)
    return f"Wrote {len(todos)} todos:\n" + run_todo_read()

def run_todo_read():
    todos = _load_todos()
    if not todos:
        return "(no todos)"
    marks = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}
    return "\n".join(f"{marks.get(t['status'], '[?]')} {t['index']}: {t['text']}" for t in todos)

def run_todo_update(index, status):
    if status not in _TODO_STATUSES:
        return f"Error: status must be one of {_TODO_STATUSES}, got {status!r}"
    todos = _load_todos()
    for t in todos:
        if t["index"] == index:
            t["status"] = status
            _save_todos(todos)
            return f"Todo #{index} -> {status}\n" + run_todo_read()
    return f"Error: no todo with index {index}"


TOOLS_LEAD += [
    _fn("todo_write",
        "Set the working todo list. Call at the start of a mapping pass to lay out your plan "
        "(e.g. one item per package/area to cover). Replaces any existing list.",
        {"items": {"type": "array", "items": {"type": "string"}}}, ["items"]),
    _fn("todo_read", "Show the current todo list with statuses.", {}, []),
    _fn("todo_update", "Update one todo's status as you make progress.",
        {"index": {"type": "integer"}, "status": {"type": "string", "enum": list(_TODO_STATUSES)}},
        ["index", "status"]),
]
DISPATCH_LEAD.update({
    "todo_write":  lambda a: run_todo_write(a["items"]),
    "todo_read":   lambda a: run_todo_read(),
    "todo_update": lambda a: run_todo_update(a["index"], a["status"]),
})
log.info("Todo tools registered.")


In [ ]:
"""
Context compaction -- lets a long mapping run stay within the context budget.
Keeps the last KEEP_RECENT messages verbatim, condenses older ones with the
summariser model, and persists the summary to MEMORY_FILE.
"""

def _estimate_size(messages):
    return sum(len(str(m.get("content", "") or "")) for m in messages)

def _render_for_summary(messages):
    parts = []
    for m in messages:
        role = m.get("role", "?")
        content = str(m.get("content", "") or "")
        if m.get("tool_calls"):
            names = [tc.get("function", {}).get("name", "?") for tc in m["tool_calls"]]
            content = (content + f"  (called tools: {', '.join(names)})").strip()
        parts.append(f"[{role}] {content}")
    return "\n\n".join(parts)

def _summarize(messages):
    transcript = _render_for_summary(messages)
    log_compact.info(f"summarising {len(messages)} msgs (~{len(transcript)} chars)")
    summary_messages = [
        {"role": "system", "content": (
            "You are a context compressor for a codebase-dependency-mapping agent. Condense the "
            "conversation into a concise summary. PRESERVE: which files/modules have already "
            "been analysed, dependencies and external connections discovered (and any still "
            "unverified), point ids worth revisiting, and what remains to cover. DROP redundant "
            "tool chatter. Write prose.")},
        {"role": "user", "content": f"Summarise this history:\n\n{transcript[:20000]}"},
    ]
    msg = ollama_chat(model=MODELS["summarizer"], messages=summary_messages, tools=None)
    return (msg.get("content") or "").strip()

def maybe_compress(messages):
    if _estimate_size(messages) < COMPRESS_THRESHOLD or len(messages) <= KEEP_RECENT + 1:
        return False
    head, body = [], messages
    if messages and messages[0].get("role") == "system":
        head, body = [messages[0]], messages[1:]
    old, recent = body[:-KEEP_RECENT], body[-KEEP_RECENT:]
    if not old:
        return False
    summary = _summarize(old)
    try:
        MEMORY_FILE.write_text(f"# Agent context memory\n\n{summary}\n", encoding="utf-8")
    except OSError:
        pass
    messages.clear()
    messages.extend(head)
    messages.append({"role": "user", "content": f"[Summary of earlier turns]:\n\n{summary}"})
    messages.append({"role": "assistant", "content": "Understood. Continuing the mapping from here."})
    messages.extend(recent)
    log_compact.info(f"compressed {len(old)} msgs; history now {len(messages)}")
    return True

log.info("Context compaction ready.")


In [ ]:
"""
Task graph -- a persistent, dependency-aware backlog for multi-session mapping.
E.g. create one task per top-level package; task_next yields the next unblocked
area to analyse. Survives restarts via TASKS_FILE.
"""

_TASK_STATUSES = ("pending", "in_progress", "done", "failed")

def _load_tasks():
    if not TASKS_FILE.exists():
        return []
    try:
        return json.loads(TASKS_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []

def _save_tasks(tasks):
    TASKS_FILE.write_text(json.dumps(tasks, indent=2), encoding="utf-8")

def run_task_create(description, depends_on=None, priority="medium"):
    tasks = _load_tasks()
    tid = uuid.uuid4().hex[:8]
    tasks.append({"id": tid, "description": description, "status": "pending",
                  "priority": priority, "depends_on": depends_on or [], "result": ""})
    _save_tasks(tasks)
    return f"Created task {tid}: {description}"

def run_task_list():
    tasks = _load_tasks()
    if not tasks:
        return "(no tasks)"
    rows = []
    for t in tasks:
        deps = f" needs={','.join(t['depends_on'])}" if t.get("depends_on") else ""
        rows.append(f"[{t['id']}] {t['status']:11s} {t['priority']:6s}{deps}  {t['description']}")
    return "\n".join(rows)

def run_task_update(task_id, status, result=""):
    if status not in _TASK_STATUSES:
        return f"Error: status must be one of {_TASK_STATUSES}, got {status!r}"
    tasks = _load_tasks()
    for t in tasks:
        if t["id"].startswith(task_id):
            t["status"] = status
            if result:
                t["result"] = result
            _save_tasks(tasks)
            return f"Task {t['id']} -> {status}"
    return f"Error: no task matching id {task_id!r}"

def run_task_next():
    tasks = _load_tasks()
    done = {t["id"] for t in tasks if t["status"] == "done"}
    for t in tasks:
        if t["status"] == "pending" and all(d in done for d in t.get("depends_on", [])):
            return f"Next: [{t['id']}] ({t['priority']}) {t['description']}"
    return "No unblocked tasks."


TOOLS_LEAD += [
    _fn("task_create", "Create a durable task (heavier than todo; for multi-session coverage).",
        {"description": {"type": "string"},
         "depends_on": {"type": "array", "items": {"type": "string"}},
         "priority": {"type": "string", "enum": ["high", "medium", "low"]}},
        ["description"]),
    _fn("task_list", "List all durable tasks.", {}, []),
    _fn("task_update", "Update a task's status (and optional result).",
        {"task_id": {"type": "string"}, "status": {"type": "string", "enum": list(_TASK_STATUSES)},
         "result": {"type": "string"}}, ["task_id", "status"]),
    _fn("task_next", "Next unblocked task (deps satisfied).", {}, []),
]
DISPATCH_LEAD.update({
    "task_create": lambda a: run_task_create(a["description"], a.get("depends_on"), a.get("priority", "medium")),
    "task_list":   lambda a: run_task_list(),
    "task_update": lambda a: run_task_update(a["task_id"], a["status"], a.get("result", "")),
    "task_next":   lambda a: run_task_next(),
})
log.info("Task graph registered.")


In [ ]:
"""
Skill loading -- optional, lazy domain playbooks under SKILLS_DIR (e.g. a
"python-imports" or "spring-beans" analysis checklist). The system prompt gets a
cheap index; load_skill pulls a full guide on demand.
"""

def _skill_dir(name):
    if not name or "/" in name or ".." in name:
        return None
    d = (SKILLS_DIR / name).resolve()
    try:
        d.relative_to(SKILLS_DIR.resolve())
    except ValueError:
        return None
    return d if (d / "SKILL.md").is_file() else None

def run_list_skills():
    if not SKILLS_DIR.is_dir():
        return "(no skills available)"
    entries = []
    for child in sorted(SKILLS_DIR.iterdir()):
        md = child / "SKILL.md"
        if not md.is_file():
            continue
        summary = ""
        for line in md.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.strip():
                summary = line.lstrip("# ").strip()
                break
        entries.append(f"- {child.name}: {summary}")
    return "\n".join(entries) if entries else "(no skills available)"

def run_load_skill(name):
    d = _skill_dir(name)
    if d is None:
        return f"Error: no skill named {name!r}. Available:\n{run_list_skills()}"
    return _truncate((d / "SKILL.md").read_text(encoding="utf-8", errors="replace"))

def skills_index_for_prompt():
    idx = run_list_skills()
    if idx == "(no skills available)":
        return ""
    return "\n\nAvailable skills (load_skill(name) to read the full guide):\n" + idx


TOOLS_LEAD += [
    _fn("list_skills", "List available skill guides.", {}, []),
    _fn("load_skill", "Load one skill guide's full text by name.",
        {"name": {"type": "string"}}, ["name"]),
]
DISPATCH_LEAD.update({
    "list_skills": lambda a: run_list_skills(),
    "load_skill":  lambda a: run_load_skill(a["name"]),
})
log.info("Skill loading registered.")


In [ ]:
"""
chat() -- the user-facing entry point. HISTORY persists across calls so a long
mapping run is one continuous conversation; reset_chat() wipes it and the memory.
The dependency map itself lives in GRAPH_FILE and persists regardless.
"""

LEAD_SYSTEM = (
    "You are a senior software architect mapping a codebase that is stored as embedded "
    "chunks in a Qdrant vector store (you cannot see the filesystem; you read the code "
    "through the Qdrant tools). Your goal: build a complete, evidence-based map of the "
    "codebase's INTERNAL dependencies and EXTERNAL connections.\n\n"
    "TOOLS:\n"
    "- Read the store: qdrant_info (CALL FIRST -- confirms schema & embedding sanity), "
    "list_files (coverage checklist), code_search (semantic), code_keyword (exhaustive "
    "substring sweep), code_scroll (walk every chunk), get_point (full code of one chunk).\n"
    "- Build the map: graph_add_edge (internal imports/calls/inherits/references/uses), "
    "graph_add_external (database/http_api/message_queue/cache/cloud_service/env_var/"
    "file_io/rpc/auth), graph_add_node, graph_summary, graph_export.\n"
    "- Plan & scale: todo_write/update (this pass), task_create/next (durable multi-session "
    "coverage), spawn_subagent (delegate one file's deep analysis -- it writes into the same "
    "map), plus file/shell and skills.\n\n"
    "METHOD:\n"
    "1. Call qdrant_info, then list_files to inventory the corpus. Write a todo/plan that "
    "covers every area; do not stop after a few files.\n"
    "2. For each file/module, read its chunks (get_point/code_scroll/code_search) and record "
    "EVERY dependency and external connection with graph_add_edge / graph_add_external, citing "
    "the exact line as evidence. Delegate deep per-file work to spawn_subagent to stay focused.\n"
    "3. Then do targeted sweeps with code_keyword for connection markers across the WHOLE corpus "
    "so nothing is missed -- e.g. 'import', 'from ', 'require(', 'requests', 'httpx', 'urllib', "
    "'psycopg2', 'sqlalchemy', 'pymongo', 'redis', 'boto3', 'google.cloud', 'azure', 'kafka', "
    "'pika', 'celery', 'grpc', 'socket', 'os.environ', 'getenv', 'open('. Adapt markers to the "
    "languages you actually see.\n"
    "4. Never invent a dependency -- only record what the code shows; if something is ambiguous, "
    "note it in details rather than guessing.\n"
    "5. Periodically call graph_summary. When coverage is complete, call graph_export('all') to "
    "write dependency_map.{json,dot,md}, then give the user a clear written summary: the main "
    "internal structure, the most-depended-on modules, and a table of external connections.\n"
    "6. When fully done, STOP calling tools and reply with that summary."
)

HISTORY: List[Dict[str, Any]] = []

def _ensure_history():
    if not HISTORY:
        system = LEAD_SYSTEM + skills_index_for_prompt()
        HISTORY.append({"role": "system", "content": system})
        log_chat.info(f"history seeded (system prompt {len(system)} chars)")

def reset_chat():
    """Wipe the conversation + memory. Does NOT delete the dependency graph; call
    new_map() for that."""
    HISTORY.clear()
    try:
        MEMORY_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_chat.info("chat history reset")

def new_map():
    """Start a fresh dependency map (deletes GRAPH_FILE)."""
    try:
        GRAPH_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_graph.info("dependency graph reset")

def chat(query: str) -> str:
    _ensure_history()
    log_chat.info(f"=== USER: {query[:200]!r} ===")
    HISTORY.append({"role": "user", "content": query})
    answer = agent_loop(HISTORY, TOOLS_LEAD, DISPATCH_LEAD, model=MODELS["lead"], label="lead")
    if maybe_compress(HISTORY):
        log_chat.info("history compacted after this turn")
    print("\n" + "=" * 70 + "\nASSISTANT:\n" + "=" * 70 + "\n" + answer)
    return answer

log.info("chat() ready. chat('...') to run; reset_chat() new conversation; new_map() fresh graph.")


In [ ]:
"""
Driver -- connectivity checks, then kick off a mapping run (guarded).

Run order: ollama healthcheck -> embedding probe -> qdrant_info. The agent run
only starts if Qdrant is reachable and the configured collection exists. If not,
it prints exactly what to fix in the config cell and skips (no hard failure).
"""
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix the config cell first."

print("\n-- embedding probe --")
try:
    v = embed_text("def connect_to_database(dsn): return psycopg2.connect(dsn)")
    print(f"{QDRANT_EMBED_MODEL} -> dim {len(v)}")
except Exception as e:
    print(f"EMBED FAILED: {e}")

print("\n-- qdrant_info --")
info = tool_qdrant_info()
print(info[:1200])

ready = not info.startswith("Error") and "not found" not in info and "WARNING" not in info
if not ready:
    print("\n>>> Qdrant not ready. Edit the config cell (QDRANT_URL / QDRANT_COLLECTION, and "
          "QDRANT_EMBED_MODEL to match how the corpus was indexed), then re-run this cell.")
    print(">>> Skipping the agent run for now.")
else:
    new_map()      # fresh graph for this demo run
    reset_chat()
    answer = chat(
        "Map this codebase. Start with qdrant_info and list_files to understand scope, plan "
        "the coverage with todo_write, then analyse the code (delegating per-file deep dives to "
        "subagents) and record every internal dependency and external connection into the graph "
        "with evidence. Run keyword sweeps for connection markers to catch anything missed. When "
        "coverage is complete, call graph_export('all') and summarise the dependency structure "
        "and the external connections for me."
    )
    print("\n" + "-" * 70)
    print("Artifacts in:", WORKDIR)
    for fn in ("dependency_map.json", "dependency_map.dot", "dependency_map.md", "dependency_graph.json"):
        p = WORKDIR / fn
        print(f"  {'OK ' if p.exists() else '-- '} {fn}" + (f"  ({p.stat().st_size} B)" if p.exists() else ""))
    print(f"history: {len(HISTORY)} msgs (~{_estimate_size(HISTORY)} chars)")
